# 1. Load and Prepare Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
import statsmodels.formula.api as smf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_poisson_deviance

df = pd.read_csv("../dataset/freMTPL2freq.csv")
df = df[df["Exposure"] > 0].copy()

df["LogDensity"] = np.log1p(df["Density"])
df["LogExposure"] = np.log(df["Exposure"])
df["ClaimOccurred"] = (df["ClaimNb"] > 0).astype(int)

age_bins = [17, 24, 34, 44, 54, 64, 74, np.inf]
age_labels = ["18-24", "25-34", "35-44", "45-54", "55-64", "65-74", "75+"]

df["DrivAgeGroup"] = pd.cut(
    df["DrivAge"],
    bins=age_bins,
    labels=age_labels,
    right=True
)

print("Shape:", df.shape)
df.head()

## 2. Train-Test Split

In [ ]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["ClaimOccurred"]
)

train_df = train_df.copy()
test_df = test_df.copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Train claim proportion:", train_df["ClaimOccurred"].mean())
print("Test claim proportion:", test_df["ClaimOccurred"].mean())

## 3. Fit Poisson Regression

In [ ]:
formula = (
    "ClaimNb ~ "
    "C(Area) + "
    "C(VehPower) + "
    "VehAge + "
    "C(DrivAgeGroup) + "
    "BonusMalus + "
    "C(VehBrand) + "
    "C(VehGas) + "
    "LogDensity + "
    "C(Region)"
)

poisson_model = smf.glm(
    formula=formula,
    data=train_df,
    family=sm.families.Poisson(),
    offset=train_df["LogExposure"]
).fit(maxiter=100)

## 4. Poisson Rate Ratios and Main Coefficient Results

In [ ]:
poisson_results = pd.DataFrame({
    "Coefficient": poisson_model.params,
    "Rate_Ratio": np.exp(poisson_model.params),
    "P_Value": poisson_model.pvalues
})

poisson_results

## 5. Observed vs Expected Zero Rates

In [ ]:
poisson_train_pred = poisson_model.predict(
    train_df,
    offset=train_df["LogExposure"]
)
poisson_train_pred = np.clip(
    np.asarray(poisson_train_pred, dtype=float),
    1e-12,
    None
)

observed_zero_rate = (train_df["ClaimNb"].to_numpy() == 0).mean()
poisson_expected_zero_rate = np.mean(np.exp(-poisson_train_pred))

zero_check = pd.DataFrame({
    "Measure": [
        "Observed zero rate",
        "Poisson expected zero rate",
        "Observed minus expected"
    ],
    "Value": [
        observed_zero_rate,
        poisson_expected_zero_rate,
        observed_zero_rate - poisson_expected_zero_rate
    ]
})

## 6. Poisson Pearson Dispersion

In [ ]:
poisson_dispersion = (
    np.sum(poisson_model.resid_pearson ** 2)
    / poisson_model.df_resid
)

## 7. Estimate Negative Binomial Dispersion Parameter Alpha

In [ ]:
y_train = train_df["ClaimNb"].to_numpy(dtype=float)
mu_train = poisson_train_pred

alpha_numerator = np.sum((y_train - mu_train) ** 2 - mu_train)
alpha_denominator = np.sum(mu_train ** 2)
alpha_hat = max(float(alpha_numerator / alpha_denominator), 1e-8)

dispersion_results = pd.DataFrame({
    "Measure": [
        "Poisson Pearson dispersion",
        "Estimated Negative Binomial alpha"
    ],
    "Value": [poisson_dispersion, alpha_hat]
})

## 8. Fit Negative Binomial Regression

In [ ]:
nb_model = smf.glm(
    formula=formula,
    data=train_df,
    family=sm.families.NegativeBinomial(alpha=alpha_hat),
    offset=train_df["LogExposure"]
).fit(maxiter=100)

## 9. Negative Binomial Rate Ratios and Main Coefficient Results

In [ ]:
nb_results = pd.DataFrame({
    "Coefficient": nb_model.params,
    "Rate_Ratio": np.exp(nb_model.params),
    "P_Value": nb_model.pvalues
})

nb_results

## 10. Poisson vs Negative Binomial Test-Set Comparison

In [ ]:
poisson_test_pred = poisson_model.predict(
    test_df,
    offset=test_df["LogExposure"]
)
nb_test_pred = nb_model.predict(
    test_df,
    offset=test_df["LogExposure"]
)

poisson_test_pred = np.clip(
    np.asarray(poisson_test_pred, dtype=float),
    1e-12,
    None
)
nb_test_pred = np.clip(
    np.asarray(nb_test_pred, dtype=float),
    1e-12,
    None
)

y_test = test_df["ClaimNb"].to_numpy(dtype=float)
test_exposure = test_df["Exposure"].to_numpy(dtype=float)

def model_metrics(name, prediction):
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        "Poisson_Deviance": mean_poisson_deviance(y_test, prediction),
        "Actual_Frequency": y_test.sum() / test_exposure.sum(),
        "Predicted_Frequency": prediction.sum() / test_exposure.sum()
    }

model_comparison = pd.DataFrame([
    model_metrics("Poisson GLM", poisson_test_pred),
    model_metrics("Negative Binomial GLM", nb_test_pred)
]).sort_values("Poisson_Deviance").reset_index(drop=True)

## 11. Poisson Five-Risk-Group Calibration

In [ ]:
poisson_calibration = test_df[["ClaimNb", "Exposure"]].copy()
poisson_calibration["Predicted_Count"] = poisson_test_pred
poisson_calibration["Predicted_Rate"] = (
    poisson_calibration["Predicted_Count"]
    / poisson_calibration["Exposure"]
)

poisson_calibration["Risk_Group"] = pd.qcut(
    poisson_calibration["Predicted_Rate"],
    q=5,
    labels=[1, 2, 3, 4, 5]
)

poisson_calibration_table = (
    poisson_calibration
    .groupby("Risk_Group", observed=True)
    .agg(
        Policies=("ClaimNb", "size"),
        Claims=("ClaimNb", "sum"),
        Exposure=("Exposure", "sum"),
        Predicted_Claims=("Predicted_Count", "sum")
    )
)

poisson_calibration_table["Observed_Frequency"] = (
    poisson_calibration_table["Claims"]
    / poisson_calibration_table["Exposure"]
)
poisson_calibration_table["Predicted_Frequency"] = (
    poisson_calibration_table["Predicted_Claims"]
    / poisson_calibration_table["Exposure"]
)

assert len(poisson_calibration_table) == 5

plt.figure(figsize=(7, 4))
plt.plot(
    poisson_calibration_table.index,
    poisson_calibration_table["Observed_Frequency"],
    marker="o",
    label="Observed"
)
plt.plot(
    poisson_calibration_table.index,
    poisson_calibration_table["Predicted_Frequency"],
    marker="o",
    label="Predicted"
)
plt.xlabel("Predicted Risk Group")
plt.ylabel("Claim Frequency")
plt.title("Poisson Calibration by Risk Group")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Final Results Summary

In [ ]:
print("ZERO CHECK")
display(zero_check)

print("DISPERSION RESULTS")
display(dispersion_results)

print("MODEL COMPARISON")
display(model_comparison)

print("POISSON CALIBRATION")
display(poisson_calibration_table)

print("Zero check: no important excess-zero problem.")
print("Dispersion check: overdispersion is present.")
print("Model decision: Poisson GLM has marginally better test metrics and remains preferred.")